# Crowd / 07 — Final test & artifact manifest

Evaluate the frozen winner on untouched 2024–2025 records, then save metrics and dependency versions.

Run every cell from top to bottom. Use the project `.venv` kernel in VS Code. No season field is used. Data and test scores are synthetic.

In [1]:
from pathlib import Path
import sys, json, inspect
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ml_service/workflows.py").exists())
sys.path.insert(0, str(ROOT))
from ml_service import workflows as w
print("Project:", ROOT)
print("Synthetic demonstration: real-world accuracy has not been measured.")

Project: /workspace/scratch/71e3a62599e6/YatraSense_Organic
Synthetic demonstration: real-world accuracy has not been measured.


In [2]:
print(inspect.getsource(w.final_test))

def final_test(task):
    df=pd.read_csv(DATA/f'{task}.csv'); te=df[df.dataset_split=='test']
    features=CROWD_FEATURES if task=='crowd' else REC_FEATURES
    model=joblib.load(MODELS/f'{task}.joblib'); pred=model.predict(te[features])
    labels=['Low','Medium','High'] if task=='crowd' else [0,1]
    report={'synthetic':True,'test_years':[2024,2025],'training_years':[2016,2022],'selection':json.loads((REPORTS/f'{task}_selection.json').read_text()),'test':metrics(te[TARGETS[task]],pred),'classification_report':classification_report(te[TARGETS[task]],pred,output_dict=True,zero_division=0),'confusion_matrix':confusion_matrix(te[TARGETS[task]],pred,labels=labels).tolist(),'class_order':labels,'features':features,'excluded_features':['visitor_count','busyness_label (target only)','row_id','request_id','dataset_split','customer_chose_destination (target only)'],'accuracy_above_80':bool(accuracy_score(te[TARGETS[task]],pred)>.8),'limitations':['Synthetic generator performance; not real cro

In [3]:
report = w.final_test("crowd")
display(pd.Series(report["test"]))
display(pd.DataFrame(report["classification_report"]).T)
display(pd.DataFrame(report["confusion_matrix"], index=report["class_order"], columns=report["class_order"]))
print("Above 80%:", report["accuracy_above_80"])
print("Saved reports:", w.REPORTS)
print("Model versions:", (w.MODELS / "versions.json").read_text())

Above 80%: False
Saved reports: /workspace/scratch/71e3a62599e6/YatraSense_Organic/reports
Model versions: {
  "python": "3.12.14",
  "scikit_learn": "1.8.0",
  "numpy": "2.3.5",
  "pandas": "2.2.3",
  "joblib": "1.5.3",
  "data_sha256": "8093405ddbc492618d99dbba9a059ee73b03949b362e71864181616f4cf9c6e7",
  "synthetic": true
}


accuracy             0.725000
macro_f1             0.718114
balanced_accuracy    0.713816
dtype: float64

,precision,recall,f1-score,support
High,0.666667,0.693069,0.679612,101.000
Low,0.789916,0.691176,0.737255,136.000
Medium,0.718750,0.757202,0.737475,243.000
accuracy,0.725000,0.725000,0.725000,0.725
macro avg,0.725111,0.713816,0.718114,480.000
weighted avg,0.727954,0.725000,0.725237,480.000


,Low,Medium,High
Low,94,42,0
Medium,24,184,35
High,1,30,70


The crowd target is relative busyness (Low/Medium/High), not a visitor-count forecast or occupancy measurement. The recommendation target is a noisy simulated binary choice. Neither task has been validated on real-world labels. New admin destinations are outside the trained catalog and are explicitly labelled unvalidated.